<a href="https://colab.research.google.com/github/angelalojko/angelalojko/blob/main/notebooks/Q2_Fine_Tuning_RNN_based_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# import gensim.downloader as api
# from gensim.models import KeyedVectors

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import re
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.utils import class_weight

# pre-processing (tokenization, numerization, and padding )
# fx. used to lowercase the sentence and strip of punctuation
def cleanText(train_text):
  return_list = []
  for i in range(len(train_text)):
    # strip of punctuation and lowercase
    cleaned_string = re.sub(r'[^\w\s]', '', train_text[i].lower())
    # strip and append to list
    return_list.append(cleaned_string.strip())
  return return_list

# tokenize each sentence to words ["this", "is", "sample"]
def tokenize(text):
  word_list = []
  for sentence in text:
    word_list.append(sentence.split())
  return word_list

# create a vocabulary set of word -> index to be used later
def buildVocab(sentence_list, dict):
  count = 2
  for sentence in sentence_list:
    for word in sentence:
      if word not in dict:
        dict[word] = count
        count = count + 1
  return dict

# this fx. converts each setence in the data set to indices ["this", "is", "sample"] -> [2,4,5]
def numerize(data_set, vocab_set):
  return_set = []
  for sentence in data_set:
    sentence_list = []
    for word in sentence:
      if word in vocab_set:
        sentence_list.append(vocab_set[word])
      else:
        sentence_list.append(vocab_set["<UNK>"])
    return_set.append(sentence_list)

  # print(return_set)
  return return_set

def padding(sentences, max_length):
  padded_list = []
  for sentence in sentences:
    # if larger then shorten the sentence
    if (len(sentence) > max_length):
      padded_sentence = sentence[:max_length]
    else:
      length_needed = max_length - len(sentence)

      padded_sentence = sentence + ([0] * length_needed)

    padded_list.append(padded_sentence)
  return padded_list


class EmotionDataset(Dataset):
  def __init__(self, sentences, emotions, polarities, empathies):
    self.sentences = torch.tensor(sentences, dtype = torch.long)
    self.emotions = torch.tensor(emotions, dtype = torch.float)
    self.polarities = torch.tensor(polarities, dtype = torch.long)
    self.empathies = torch.tensor(empathies, dtype = torch.float)

  def __len__(self):
    return(len(self.sentences))

  def __getitem__(self, id):
    return (self.sentences[id], self.emotions[id], self.polarities[id], self.empathies[id])


class EmotionRNN(nn.Module):
  def __init__(self, input_size, embed_dim, hidden_size, class_num, dropout=0.3):
    super(EmotionRNN, self).__init__()
    # set the drop out
    self.dropout = nn.Dropout(dropout)
    # set embedding and ltsm
    self.embedding = nn.Embedding(input_size, embed_dim, padding_idx=0)
    self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
    # set three sep. heads for model
    self.emotion_head = nn.Linear(hidden_size, 1)
    self.polarity_head = nn.Linear(hidden_size, class_num)
    self.empathy_head = nn.Linear(hidden_size, 1)

  def forward(self,x):
    embedding = self.embedding(x)
    lstm_out, (hidden, cell) = self.lstm(embedding)
    hidden_state = hidden.squeeze(0)
    hidden_state = self.dropout(hidden_state)
    # creating 3 separate head predictions
    emo_out = self.emotion_head(hidden_state).squeeze(1)
    po_out = self.polarity_head(hidden_state)
    emp_out = self.empathy_head(hidden_state).squeeze(1)

    return emo_out, po_out, emp_out

# fx used for training the model
def train_model(model, train_loader, data_loader, device, weights):
  mse_loss = nn.MSELoss()
  cross_entropy_loss = nn.CrossEntropyLoss(weight= weights.to(device))
  optimizer = torch.optim.Adam(model.parameters(), lr= .001)

  for epoch in range(10):
    model.train()
    total_loss = 0

    for sentence, emotion, polarity, empathy in train_loader:
      optimizer.zero_grad()
      sentence = sentence.to(device)
      emotion = emotion.to(device)
      polarity = polarity.to(device)
      empathy = empathy.to(device)

      emotion_pred, polarity_pred, empathy_pred = model(sentence)

      emotion_loss = mse_loss(emotion_pred, emotion)
      polarity_loss = cross_entropy_loss(polarity_pred, polarity)
      empathy_loss = mse_loss(empathy_pred, empathy)

      loss_batch = emotion_loss + polarity_loss + empathy_loss
      loss_batch.backward()
      optimizer.step()
      total_loss += loss_batch.item()
    # getting the average loss and print
    average_loss = total_loss/ len(train_loader)
    print(f"Epoch {epoch+1}/{10} - Loss: {average_loss:.3f}")

  return model

# using dev loader
def evaluation(model, device,  dev_loader):
  model.eval()

  all_emotion, all_polarity, all_empathy = [] , [] , []
  pred_emotion, pred_polarity, pred_empathy = [], [], []

  with torch.no_grad():
    for sentence, emt, pol, emp in dev_loader:
      sentence = sentence.to(device)

      emotion_pred, polarity_pred, empathy_pred = model(sentence)

      all_emotion.extend(emt.tolist())
      all_polarity.extend(pol.tolist())
      all_empathy.extend(emp.tolist())

      pred_emotion.extend(emotion_pred.detach().cpu().tolist())
      pred_polarity.extend(polarity_pred.argmax(dim=1).detach().cpu().tolist())
      pred_empathy.extend(empathy_pred.detach().cpu().tolist())

    mae_emotion = mean_absolute_error(all_emotion, pred_emotion)
    print(f"MAE Emotion: {mae_emotion:.3f}")
    mae_empathy = mean_absolute_error(all_empathy, pred_empathy)
    print(f"MAE Emapthy: {mae_empathy:.3f}")
    report = classification_report(all_polarity, pred_polarity, zero_division=0)
    print("Polarity Report: \n")
    print(report)


def predict(model, device,  test_loader, test_df):
  model.eval()

  pred_emotion, pred_polarity, pred_empathy = [], [], []

  with torch.no_grad():
    for sentence, _, _, _ in test_loader:
      sentence = sentence.to(device)

      emotion_pred, polarity_pred, empathy_pred = model(sentence)


      pred_emotion.extend(emotion_pred.detach().cpu().tolist())
      pred_polarity.extend(polarity_pred.argmax(dim=1).detach().cpu().tolist())
      pred_empathy.extend(empathy_pred.detach().cpu().tolist())

  predictions_df = pd.DataFrame({
    'id': test_df['id'].values,
    'Emotion': pred_emotion,
    'EmotionalPolarity': pred_polarity,
    'Empathy': pred_empathy
  })

  predictions_df.to_csv("predictions.csv", index=False)
  print(predictions_df.head())

def main():
  # loading the datasets
  train_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_train.csv')
  test_df =  pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_test.csv', on_bad_lines='skip')
  dev_df =  pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_dev.csv', on_bad_lines='skip')

  # print(train_df.head())

  # split the data set into texts for preprocessing
  train_text = train_df["text"].astype(str).tolist()
  test_text = test_df["text"].astype(str).tolist()
  dev_text = dev_df["text"].astype(str).tolist()

  # train id,  emotion, polarity, and empathy
  train_id = train_df["id"].astype(int).tolist()
  train_emotion = train_df["Emotion"].astype(float).tolist()
  train_polarity = train_df["EmotionalPolarity"].astype(int).tolist()
  train_empathy = train_df["Empathy"].astype(float).tolist()

  # dev id,  emotion, polarity, and empathy
  dev_id = dev_df["id"].astype(int).tolist()
  dev_emotion = dev_df["Emotion"].astype(float).tolist()
  dev_polarity = dev_df["EmotionalPolarity"].astype(int).tolist()
  dev_empathy = dev_df["Empathy"].astype(float).tolist()

  cleaned_train = cleanText(train_text)
  tokenized_train_list = tokenize(cleaned_train)
  # print(tokenized_sentence_list)

  # processing the test set
  cleaned_test = cleanText(test_text)
  tokenized_test_list = tokenize(cleaned_test)
  # processing the dev set
  cleaned_dev = cleanText(dev_text)
  tokenized_dev_list = tokenize(cleaned_dev)

  # building a vocab set for normalization
  vocabulary_set = {"<PAD>" : 0 , "<UNK>": 1}
  buildVocab(tokenized_train_list, vocabulary_set)
  vocab_size = len(vocabulary_set)
  train_text_to_index = numerize(tokenized_train_list, vocabulary_set)
  test_text_to_index = numerize(tokenized_test_list, vocabulary_set)
  dev_text_to_index = numerize(tokenized_dev_list, vocabulary_set)



  # print(train_text_to_index)
  # print(test_text_to_index)
  # print(dev_text_to_index)

  # find the appropriate length for padding the dataframes
  train_length = []
  for sentence in train_text_to_index:
    train_length.append(len(sentence))

  # print("max is: ", max(train_length))
  # print("90% is: ", np.percentile(train_length, 90))
  # print("95% is: ", np.percentile(train_length, 95))
  # print("99% is: ", np.percentile(train_length, 99))
  chosen_length = 45

  # padding data using the chosen length
  padded_train = padding(train_text_to_index, chosen_length)
  padded_test = padding(test_text_to_index, chosen_length)
  padded_dev = padding(dev_text_to_index, chosen_length)
  # batching the train and dev data sets
  train_dataset = EmotionDataset(padded_train, train_emotion, train_polarity, train_empathy)
  train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  dev_dataset = EmotionDataset(padded_dev, dev_emotion, dev_polarity, dev_empathy)
  dev_dataloader = DataLoader(dev_dataset, batch_size=32, shuffle=False)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model = EmotionRNN(vocab_size, embed_dim= 64, hidden_size= 128, class_num= 4, dropout=0.3).to(device)
  # calculate class weight to handle data imbalance to ensure results are less skewed
  weights = class_weight.compute_class_weight('balanced', classes = np.unique(train_polarity), y = train_polarity)
  weights = torch.tensor(weights).float()

  trained_model = train_model(model, train_dataloader, dev_dataloader, device, weights)
  evaluation(trained_model, device, dev_dataloader)

  test_dataset = EmotionDataset(padded_test, [0]*len(padded_test), [0]*len(padded_test), [0]*len(padded_test))
  test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle=False)

  predict(trained_model, device, test_dataloader, test_df)

main()

Epoch 1/10 - Loss: 3.226
Epoch 2/10 - Loss: 2.800
Epoch 3/10 - Loss: 2.637
Epoch 4/10 - Loss: 2.318
Epoch 5/10 - Loss: 2.098
Epoch 6/10 - Loss: 1.968
Epoch 7/10 - Loss: 1.890
Epoch 8/10 - Loss: 1.803
Epoch 9/10 - Loss: 1.725
Epoch 10/10 - Loss: 1.623
MAE Emotion: 0.556
MAE Emapthy: 0.785
Polarity Report: 

              precision    recall  f1-score   support

           0       0.35      0.31      0.33       170
           1       0.68      0.48      0.57       460
           2       0.55      0.78      0.64       357

    accuracy                           0.56       987
   macro avg       0.52      0.52      0.51       987
weighted avg       0.57      0.56      0.55       987

   id   Emotion  EmotionalPolarity   Empathy
0   1  2.414292                  2  2.334256
1   2  2.123235                  1  1.998827
2   3  2.764949                  2  2.735326
3   4  1.645535                  1  1.541310
4   5  2.387851                  2  2.306699
